# Railway Track Defect Classifier — Explainable, Calibrated, Priority-Ranked

**Run everything in this notebook, top to bottom, on a Colab GPU runtime.** All compute happens here — you don't need anything locally except a Kaggle API key (`kaggle.json`) to download the dataset, and a browser at the end to download your results.

**What this builds:**
1. A binary (defective / non-defective) image classifier, transfer-learned from a pretrained backbone (~10-20 min)
2. Grad-CAM visual explanations (where the model thinks the defect is)
3. Confidence calibration (temperature scaling + Expected Calibration Error)
4. Monte Carlo Dropout uncertainty estimation
5. A Grad-CAM-derived severity proxy (how much of the image is affected)
6. A combined inspection-priority score (severity x uncertainty)
7. Selective prediction (reject/flag low-confidence cases instead of forcing an answer)
8. A Grad-CAM sanity check (proves the explanations are real, not decoration)

All of 2-8 above are inference-time / post-processing on the one trained model from step 1 — no extra training, no extra GPU cost.

**Before you start:** Runtime > Change runtime type > GPU.


## 1. Setup

Installs and imports. Run both cells.

In [ ]:
!pip -q install kaggle
import torch, torchvision, numpy as np, random, os, json, shutil
from pathlib import Path
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU - Runtime > Change runtime type > GPU, then re-run.")
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
WORK = Path("/content/drive/MyDrive/RailDefectClassifier")
WORK.mkdir(parents=True, exist_ok=True)
(WORK / "outputs").mkdir(exist_ok=True)
print("Outputs will be saved to", WORK)


## 2. Kaggle dataset download

**What to do first (one-time, outside Colab):**
1. Go to kaggle.com > your profile picture > **Settings** > **API** > **Create New Token**. This downloads a `kaggle.json` file.
2. Run the cell below, then click "Choose Files" and upload that `kaggle.json`.


In [ ]:
from google.colab import files
print("Upload your kaggle.json now:")
uploaded = files.upload()
os.makedirs("/root/.kaggle", exist_ok=True)
shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("Kaggle credentials installed.")


In [ ]:
DATA_DIR = Path("/content/railway_data")
DATA_DIR.mkdir(exist_ok=True)
!kaggle datasets download -d salmaneunus/railway-track-fault-detection -p "{DATA_DIR}" --unzip
print("Downloaded. Top-level contents:")
for p in sorted(DATA_DIR.iterdir()):
    print(" ", p.name, "(dir)" if p.is_dir() else "(file)")


## 3-4. Locate the dataset structure and build train/val/test lists

This dataset is usually shipped as a wrapper folder containing its OWN `Train` / `Validation` / `Test` split, each with two class subfolders (`Defective` / `Non defective`). The cell below looks for that layout first and uses it directly (so we respect whatever split the dataset authors intended, rather than silently repooling and re-splitting it ourselves). If that layout isn't found, it falls back to finding one pair of class folders and does a fresh stratified split.

**Check the printed image counts look sensible (hundreds+ per split, not a couple dozen) before continuing.**


In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
SPLIT_ALIASES = {"train": "train", "training": "train",
                  "val": "val", "validation": "val", "valid": "val",
                  "test": "test", "testing": "test"}

def find_split_layout(root, max_depth=5):
    """Search for a directory whose children include >=2 recognizable split names
    (train/val/test, however named/cased), each itself containing exactly 2 class subfolders."""
    root = Path(root)
    frontier = [root]
    for _ in range(max_depth):
        nxt = []
        for d in frontier:
            try:
                kids = [k for k in sorted(d.iterdir()) if k.is_dir() and not k.name.startswith(".")]
            except OSError:
                continue
            found = {}
            for k in kids:
                key = SPLIT_ALIASES.get(k.name.strip().lower())
                if key:
                    found[key] = k
            if len(found) >= 2:
                return found
            nxt.extend(kids)
        frontier = nxt
    return None

def pick_class_dirs(split_dir):
    """Within one split folder, find its 2 class subfolders and order them [defective, non-defective]."""
    kids = [k for k in sorted(split_dir.iterdir()) if k.is_dir()]
    assert len(kids) == 2, f"expected 2 class folders under {split_dir}, found {len(kids)}: {kids}"
    names_lower = [k.name.lower() for k in kids]
    if "non" in names_lower[0] and "non" not in names_lower[1]:
        kids = [kids[1], kids[0]]
    return kids  # [defective_dir, non_defective_dir]

def collect(folder, label):
    return [(p, label) for p in Path(folder).rglob("*") if p.suffix.lower() in IMG_EXTS]

layout = find_split_layout(DATA_DIR)

if layout and len(layout) >= 2:
    print("Found the dataset's own split folders:", {k: str(v) for k, v in layout.items()})
    split_items = {}
    for split in ("train", "val", "test"):
        if split not in layout:
            split_items[split] = []
            continue
        def_dir, nondef_dir = pick_class_dirs(layout[split])
        split_items[split] = collect(def_dir, 1) + collect(nondef_dir, 0)
    # if there's no separate val split, carve one out of train
    if not split_items.get("val"):
        from sklearn.model_selection import train_test_split as _tts
        tr = split_items["train"]
        tr_paths, tr_labels = [p for p, _ in tr], [l for _, l in tr]
        tr_p2, val_p2, tr_y2, val_y2 = _tts(tr_paths, tr_labels, test_size=0.15, stratify=tr_labels, random_state=SEED)
        split_items["train"] = list(zip(tr_p2, tr_y2))
        split_items["val"] = list(zip(val_p2, val_y2))
    train_p = [p for p, _ in split_items["train"]]; train_y = [l for _, l in split_items["train"]]
    val_p   = [p for p, _ in split_items["val"]];   val_y   = [l for _, l in split_items["val"]]
    test_p  = [p for p, _ in split_items["test"]];  test_y  = [l for _, l in split_items["test"]]
else:
    print("No recognizable Train/Val/Test layout found - falling back to: find one pair of class folders, pool, and re-split.")
    def find_class_folders(root, max_depth=4):
        root = Path(root)
        frontier = [root]
        for _ in range(max_depth):
            nxt = []
            for d in frontier:
                try:
                    kids = [k for k in sorted(d.iterdir()) if k.is_dir() and not k.name.startswith(".")]
                except OSError:
                    continue
                if len(kids) == 2:
                    return kids
                nxt.extend(kids)
            frontier = nxt
        return None
    CLASS_DIRS = find_class_folders(DATA_DIR)
    assert CLASS_DIRS is not None, "Could not find class folders either way - print the tree with !find and set CLASS_DIRS manually."
    names_lower = [d.name.lower() for d in CLASS_DIRS]
    if "non" in names_lower[0] and "non" not in names_lower[1]:
        CLASS_DIRS = [CLASS_DIRS[1], CLASS_DIRS[0]]
    items = collect(CLASS_DIRS[0], 1) + collect(CLASS_DIRS[1], 0)
    paths = [p for p, _ in items]; labels = [l for _, l in items]
    from sklearn.model_selection import train_test_split
    train_p, temp_p, train_y, temp_y = train_test_split(paths, labels, test_size=0.3, stratify=labels, random_state=SEED)
    val_p, test_p, val_y, test_y = train_test_split(temp_p, temp_y, test_size=0.5, stratify=temp_y, random_state=SEED)

print(f"train={len(train_p)} (defective={sum(train_y)})  val={len(val_p)} (defective={sum(val_y)})  test={len(test_p)} (defective={sum(test_y)})")
assert len(train_p) > 100, "Train set looks too small - something's still wrong with the detected folders, check above."


## 5. Dataset / DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

IMG_SIZE = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # ImageNet stats (matches the pretrained backbone)

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class RailDataset(Dataset):
    def __init__(self, paths, labels, tfm):
        self.paths, self.labels, self.tfm = paths, labels, tfm
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tfm(img), self.labels[i], str(self.paths[i])

BATCH = 32
train_ds = RailDataset(train_p, train_y, train_tfms)
val_ds   = RailDataset(val_p, val_y, eval_tfms)
test_ds  = RailDataset(test_p, test_y, eval_tfms)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2)
val_dl   = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2)
test_dl  = DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=2)
print("batches - train:", len(train_dl), "val:", len(val_dl), "test:", len(test_dl))


## 6. Model

ResNet18 pretrained on ImageNet, fine-tuned. A Dropout layer sits before the final linear layer - used normally during training, and deliberately left *on* during inference later for Monte Carlo Dropout uncertainty (section 10).

In [ ]:
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

def build_model():
    m = resnet18(weights=ResNet18_Weights.DEFAULT)
    in_features = m.fc.in_features
    m.fc = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(in_features, 2))   # 2 logits: [non-defective, defective]
    return m.to(DEVICE)

model = build_model()
print("Model ready.")


## 7. Training

~10-20 min on a Colab T4 for this dataset size. Watch train/val loss and accuracy printed each epoch.

In [ ]:
import torch.optim as optim

EPOCHS = 12
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def run_epoch(dl, train):
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    for x, y, _ in dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.set_grad_enabled(train):
            out = model(x)
            loss = criterion(out, y)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
        n += x.size(0)
    return total_loss / n, correct / n

best_val_acc = 0.0
BEST_PATH = WORK / "outputs" / "best_model.pt"
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_dl, train=True)
    val_loss, val_acc = run_epoch(val_dl, train=False)
    scheduler.step()
    history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
    history["val_loss"].append(val_loss); history["val_acc"].append(val_acc)
    print(f"epoch {epoch}/{EPOCHS}  train_loss={tr_loss:.4f} train_acc={tr_acc:.3f}  val_loss={val_loss:.4f} val_acc={val_acc:.3f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), BEST_PATH)
print("Best val accuracy:", best_val_acc, "| saved to", BEST_PATH)
model.load_state_dict(torch.load(BEST_PATH))


### Training curves

In [ ]:
epochs_range = list(range(1, EPOCHS + 1))
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].plot(epochs_range, history["train_loss"], label="train"); axs[0].plot(epochs_range, history["val_loss"], label="val")
axs[0].set_title("Loss"); axs[0].set_xlabel("epoch"); axs[0].legend()
axs[1].plot(epochs_range, history["train_acc"], label="train"); axs[1].plot(epochs_range, history["val_acc"], label="val")
axs[1].set_title("Accuracy"); axs[1].set_xlabel("epoch"); axs[1].legend()
plt.tight_layout()
(WORK / "outputs" / "visualizations").mkdir(parents=True, exist_ok=True)
plt.savefig(WORK / "outputs" / "visualizations" / "training_curves.png", dpi=120)
plt.show()


## 8. Evaluation on test set

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve

model.eval()
all_logits, all_y, all_paths = [], [], []
with torch.no_grad():
    for x, y, p in test_dl:
        out = model(x.to(DEVICE))
        all_logits.append(out.cpu())
        all_y.extend(y.tolist())
        all_paths.extend(p)
all_logits = torch.cat(all_logits)
all_probs = torch.softmax(all_logits, dim=1)[:, 1].numpy()   # P(defective)
all_preds = (all_probs >= 0.5).astype(int)
all_y = np.array(all_y)

test_acc = accuracy_score(all_y, all_preds)
cm = confusion_matrix(all_y, all_preds)
roc_auc = roc_auc_score(all_y, all_probs)
fpr, tpr, roc_thresholds = roc_curve(all_y, all_probs)

print("Test accuracy:", round(test_acc, 4))
print("\nConfusion matrix [[TN, FP], [FN, TP]]:\n", cm)
print("\n", classification_report(all_y, all_preds, target_names=["Non-defective", "Defective"]))
print("ROC-AUC:", round(roc_auc, 4))

### Confusion matrix + ROC curve (plots)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4.5))

im = axs[0].imshow(cm, cmap="Blues")
axs[0].set_xticks([0, 1]); axs[0].set_xticklabels(["Non-defective", "Defective"])
axs[0].set_yticks([0, 1]); axs[0].set_yticklabels(["Non-defective", "Defective"])
axs[0].set_xlabel("predicted"); axs[0].set_ylabel("true"); axs[0].set_title("Confusion matrix")
for i in range(2):
    for j in range(2):
        axs[0].text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")

axs[1].plot(fpr, tpr, label=f"ROC (AUC={roc_auc:.3f})")
axs[1].plot([0, 1], [0, 1], "--", color="gray", label="chance")
axs[1].set_xlabel("False Positive Rate"); axs[1].set_ylabel("True Positive Rate")
axs[1].set_title("ROC curve"); axs[1].legend()

plt.tight_layout()
plt.savefig(WORK / "outputs" / "visualizations" / "confusion_matrix_roc.png", dpi=120)
plt.show()


## 9. Grad-CAM — where the model thinks the defect is

Manual implementation via forward/backward hooks on the last conv block (`layer4`) - no extra library needed, and gives us full control for the sanity check in section 12.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def __call__(self, x, class_idx=None):
        self.model.eval()
        out = self.model(x)
        if class_idx is None:
            class_idx = out.argmax(1).item()
        self.model.zero_grad()
        out[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)   # global-average-pool the gradients
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)
        cam = torch.nn.functional.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx, torch.softmax(out, dim=1).detach().cpu().numpy()[0]

gradcam = GradCAM(model, model.layer4[-1])
print("Grad-CAM ready (hooked on layer4).")


In [ ]:
import matplotlib.pyplot as plt

def unnormalize(t):
    img = t.clone().cpu().numpy().transpose(1, 2, 0)
    img = img * np.array(STD) + np.array(MEAN)
    return np.clip(img, 0, 1)

def show_gradcam_examples(n=6):
    idxs = random.sample(range(len(test_ds)), n)
    fig, axs = plt.subplots(2, n, figsize=(3 * n, 6))
    for i, idx in enumerate(idxs):
        x, y, path = test_ds[idx]
        xb = x.unsqueeze(0).to(DEVICE)
        cam, pred_class, probs = gradcam(xb)
        img = unnormalize(x)
        axs[0, i].imshow(img); axs[0, i].axis("off")
        axs[0, i].set_title(f"true={'Def' if y else 'OK'} pred={'Def' if pred_class else 'OK'}", fontsize=8)
        axs[1, i].imshow(img); axs[1, i].imshow(cam, cmap="jet", alpha=0.45); axs[1, i].axis("off")
        axs[1, i].set_title(f"P(defective)={probs[1]:.2f}", fontsize=8)
    plt.suptitle("Top: original (true vs predicted label). Bottom: Grad-CAM overlay (red = influential region)")
    plt.tight_layout()
    (WORK / "outputs" / "visualizations").mkdir(parents=True, exist_ok=True)
    plt.savefig(WORK / "outputs" / "visualizations" / "gradcam_examples.png", dpi=120)
    plt.show()

show_gradcam_examples()


## 10. Confidence calibration — temperature scaling + Expected Calibration Error

Raw softmax confidence is often overconfident. We fit one scalar (the "temperature") on the validation set to soften/sharpen the logits so confidence better matches actual accuracy, then report Expected Calibration Error (ECE) before and after.

In [ ]:
def expected_calibration_error(confidences, correctness, n_bins=10):
    confidences, correctness = np.asarray(confidences), np.asarray(correctness)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
        if mask.sum() == 0:
            continue
        bin_acc = correctness[mask].mean()
        bin_conf = confidences[mask].mean()
        ece += (mask.sum() / len(confidences)) * abs(bin_acc - bin_conf)
    return ece

# collect validation logits for temperature fitting
model.eval()
val_logits, val_y_ = [], []
with torch.no_grad():
    for x, y, _ in val_dl:
        val_logits.append(model(x.to(DEVICE)).cpu())
        val_y_.extend(y.tolist())
val_logits = torch.cat(val_logits)
val_y_ = torch.tensor(val_y_)

temperature = nn.Parameter(torch.ones(1) * 1.5)
opt_t = optim.LBFGS([temperature], lr=0.01, max_iter=200)
nll = nn.CrossEntropyLoss()

def _closure():
    opt_t.zero_grad()
    loss = nll(val_logits / temperature, val_y_)
    loss.backward()
    return loss
opt_t.step(_closure)
T = temperature.item()
print("Fitted temperature:", round(T, 3))

# before/after on the TEST set
raw_conf = torch.softmax(all_logits, dim=1).max(1).values.numpy()
cal_conf = torch.softmax(all_logits / T, dim=1).max(1).values.numpy()
correctness = (all_preds == all_y).astype(int)

ece_before = expected_calibration_error(raw_conf, correctness)
ece_after = expected_calibration_error(cal_conf, correctness)
print(f"ECE before calibration: {ece_before:.4f}")
print(f"ECE after calibration:  {ece_after:.4f}")


def reliability_bins(confidences, correctness, n_bins=10):
    confidences, correctness = np.asarray(confidences), np.asarray(correctness)
    edges = np.linspace(0, 1, n_bins + 1)
    bin_conf, bin_acc, bin_count = [], [], []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
        bin_conf.append(float(confidences[mask].mean()) if mask.sum() else None)
        bin_acc.append(float(correctness[mask].mean()) if mask.sum() else None)
        bin_count.append(int(mask.sum()))
    return edges.tolist(), bin_conf, bin_acc, bin_count

bin_edges, raw_bin_conf, raw_bin_acc, raw_bin_count = reliability_bins(raw_conf, correctness)
_, cal_bin_conf, cal_bin_acc, cal_bin_count = reliability_bins(cal_conf, correctness)


### Calibration reliability diagram

In [ ]:
def plot_reliability(ax, bin_conf, bin_acc, title):
    xs = [c for c in bin_conf if c is not None]
    ys = [a for c, a in zip(bin_conf, bin_acc) if c is not None]
    ax.bar(xs, ys, width=0.08, alpha=0.7, edgecolor="black", label="observed")
    ax.plot([0, 1], [0, 1], "--", color="gray", label="perfect calibration")
    ax.set_xlabel("confidence"); ax.set_ylabel("accuracy"); ax.set_title(title); ax.legend()

fig, axs = plt.subplots(1, 2, figsize=(11, 4.5))
plot_reliability(axs[0], raw_bin_conf, raw_bin_acc, f"Before calibration (ECE={ece_before:.3f})")
plot_reliability(axs[1], cal_bin_conf, cal_bin_acc, f"After calibration (ECE={ece_after:.3f})")
plt.tight_layout()
plt.savefig(WORK / "outputs" / "visualizations" / "reliability_diagram.png", dpi=120)
plt.show()


## 11. Monte Carlo Dropout uncertainty

Leave dropout *on* at inference (instead of the usual eval-mode off) and run several stochastic forward passes per image. High variance across passes = the model is uncertain about that image, independent of the calibrated confidence score.

In [ ]:
def enable_dropout(m):
    for module in m.modules():
        if isinstance(module, nn.Dropout):
            module.train()

N_PASSES = 15
model.eval()
enable_dropout(model)   # keep dropout active even though the rest of the model is in eval mode

mc_probs = []   # [N_PASSES, n_test]
with torch.no_grad():
    for _ in range(N_PASSES):
        pass_probs = []
        for x, y, p in test_dl:
            out = model(x.to(DEVICE))
            pass_probs.append(torch.softmax(out, dim=1)[:, 1].cpu())
        mc_probs.append(torch.cat(pass_probs).numpy())
mc_probs = np.stack(mc_probs)   # [N_PASSES, n_test]
mc_mean = mc_probs.mean(axis=0)
mc_std = mc_probs.std(axis=0)   # epistemic uncertainty per test image
print("MC Dropout done. Mean P(defective) uncertainty (std) - min/median/max:",
      round(mc_std.min(), 4), round(np.median(mc_std), 4), round(mc_std.max(), 4))
model.eval()   # back to normal eval mode (dropout off) for the rest of the notebook


In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(mc_std, bins=30, edgecolor="black")
plt.xlabel("MC-Dropout std (epistemic uncertainty)"); plt.ylabel("count of test images")
plt.title("Distribution of model uncertainty across test images")
plt.tight_layout()
plt.savefig(WORK / "outputs" / "visualizations" / "mc_dropout_uncertainty_hist.png", dpi=120)
plt.show()


### 11b. Aleatoric vs. epistemic uncertainty decomposition

The MC-Dropout std above is a single blended uncertainty number. We can split it into two meaningfully different things, using the standard decomposition (Kendall & Gal, *"What Uncertainties Do We Need in Bayesian Deep Learning for Computer Vision?"*, NeurIPS 2017; the epistemic term below is the same quantity as the BALD acquisition score, Houlsby et al. 2011):

- **Aleatoric uncertainty** — irreducible noise in the data itself (blurry photo, ambiguous defect) — the model would stay unsure about this even with infinite training data.
- **Epistemic uncertainty** — uncertainty from the model not having seen enough similar examples — this is the part more training data could actually fix.

Pure post-processing on the MC-Dropout passes already computed above - no extra forward/backward passes.


In [ ]:
EPS = 1e-8

def binary_entropy(p):
    p = np.clip(p, EPS, 1 - EPS)
    return -(p * np.log(p) + (1 - p) * np.log(1 - p))

total_uncertainty = binary_entropy(mc_mean)                 # H[ E[p] ]  - uncertainty of the averaged prediction
aleatoric_uncertainty = binary_entropy(mc_probs).mean(axis=0)   # E[ H[p] ] - average uncertainty across passes
epistemic_uncertainty = np.clip(total_uncertainty - aleatoric_uncertainty, 0, None)   # mutual information (BALD score)

print("Total uncertainty      - min/median/max:", round(total_uncertainty.min(), 4), round(np.median(total_uncertainty), 4), round(total_uncertainty.max(), 4))
print("Aleatoric uncertainty   - min/median/max:", round(aleatoric_uncertainty.min(), 4), round(np.median(aleatoric_uncertainty), 4), round(aleatoric_uncertainty.max(), 4))
print("Epistemic uncertainty   - min/median/max:", round(epistemic_uncertainty.min(), 4), round(np.median(epistemic_uncertainty), 4), round(epistemic_uncertainty.max(), 4))

fig, axs = plt.subplots(1, 2, figsize=(11, 4.5))
axs[0].hist(aleatoric_uncertainty, bins=25, alpha=0.7, label="aleatoric", color="tab:orange")
axs[0].hist(epistemic_uncertainty, bins=25, alpha=0.7, label="epistemic", color="tab:blue")
axs[0].set_xlabel("uncertainty"); axs[0].set_ylabel("count"); axs[0].legend(); axs[0].set_title("Aleatoric vs. epistemic distributions")
axs[1].scatter(aleatoric_uncertainty, epistemic_uncertainty, alpha=0.4)
axs[1].set_xlabel("aleatoric"); axs[1].set_ylabel("epistemic"); axs[1].set_title("Per-image: data noise vs. model unfamiliarity")
plt.tight_layout()
(WORK / "outputs" / "visualizations").mkdir(parents=True, exist_ok=True)
plt.savefig(WORK / "outputs" / "visualizations" / "uncertainty_decomposition.png", dpi=120)
plt.show()


### 11c. Out-of-distribution flagging (Mahalanobis distance in feature space)

A model can be confidently wrong when it sees something nothing like its training data (a different camera angle, a completely different object). We check for that directly: extract the penultimate-layer embedding (the 512-d feature vector right before the final classifier) for every training image once, fit a Gaussian to that embedding distribution, then measure how far any new image's embedding sits from it (Mahalanobis distance). A large distance means "this doesn't look like anything the model trained on" - a trust signal that is independent of the model's own confidence score. Reference: Lee et al., *"A Simple Unified Framework for Detecting Out-of-Distribution Samples,"* NeurIPS 2018.

One forward pass per image to grab the embedding (no backward pass, no retraining) - cheap. Here we compute it for train (to fit the reference distribution) and test (to see where real test images land relative to it); a genuinely out-of-distribution example would need external data not included in this dataset, so treat the test-set numbers here as a relative/diagnostic ranking, not a demonstration of catching true OOD inputs.


In [ ]:
feat_store = {}
def _feature_hook(module, inp, out):
    feat_store["feat"] = out.detach()

hook_handle = model.avgpool.register_forward_hook(_feature_hook)

def get_embeddings(dl):
    embeds = []
    model.eval()
    with torch.no_grad():
        for x, y, p in dl:
            _ = model(x.to(DEVICE))
            embeds.append(feat_store["feat"].squeeze(-1).squeeze(-1).cpu().numpy())
    return np.concatenate(embeds, axis=0)

train_embeds = get_embeddings(train_dl)
test_embeds = get_embeddings(test_dl)
hook_handle.remove()

mu = train_embeds.mean(axis=0)
cov = np.cov(train_embeds, rowvar=False)
cov_inv = np.linalg.pinv(cov + np.eye(cov.shape[0]) * 1e-6)   # regularized for numerical stability

def mahalanobis(embeds, mu, cov_inv):
    diff = embeds - mu
    return np.sqrt(np.clip(np.einsum("bi,ij,bj->b", diff, cov_inv, diff), 0, None))

train_maha = mahalanobis(train_embeds, mu, cov_inv)
test_maha = mahalanobis(test_embeds, mu, cov_inv)

print("Train Mahalanobis distance (reference) - median:", round(float(np.median(train_maha)), 2))
print("Test Mahalanobis distance               - median:", round(float(np.median(test_maha)), 2),
      "| max:", round(float(test_maha.max()), 2))

plt.figure(figsize=(7, 4.5))
plt.hist(train_maha, bins=30, alpha=0.6, label="train (reference)", color="tab:green")
plt.hist(test_maha, bins=30, alpha=0.6, label="test", color="tab:purple")
plt.xlabel("Mahalanobis distance from train embedding distribution"); plt.ylabel("count")
plt.title("Out-of-distribution signal: test images vs. the training distribution")
plt.legend()
plt.tight_layout()
plt.savefig(WORK / "outputs" / "visualizations" / "ood_mahalanobis.png", dpi=120)
plt.show()

top_ood_idx = np.argsort(-test_maha)[:5]
print("\n5 test images furthest from the training distribution (worth a manual look):")
for i in top_ood_idx:
    print(f"  {all_paths[i]}  (distance={test_maha[i]:.2f}, true={all_y[i]}, pred={all_preds[i]})")


## 12. Grad-CAM severity proxy + combined inspection-priority score

Severity proxy: fraction of the image where the Grad-CAM heatmap is strongly activated (>50% of its own max) - a bigger activated region suggests a more extensive visible defect. Priority score combines severity with how uncertain the model is (both the calibration-based and MC-Dropout-based signals), so severe-looking AND uncertain images get flagged first for human review.

In [ ]:
import hashlib

def severity_proxy(cam, threshold=0.5):
    return float((cam > threshold).mean())   # fraction of pixels strongly activated

TEST_IMG_DIR = WORK / "outputs" / "test_images"
GRADCAM_DIR = WORK / "outputs" / "gradcam_overlays"
TEST_IMG_DIR.mkdir(parents=True, exist_ok=True)
GRADCAM_DIR.mkdir(parents=True, exist_ok=True)

def safe_name(path, i):
    # short, stable, collision-free filename for the export (original filenames may repeat across sources)
    h = hashlib.md5(str(path).encode()).hexdigest()[:8]
    return f"{i:04d}_{h}{Path(path).suffix.lower()}"

results = []
model.eval()
for i in range(len(test_ds)):
    x, y, path = test_ds[i]
    xb = x.unsqueeze(0).to(DEVICE)
    cam, pred_class, probs = gradcam(xb)
    sev = severity_proxy(cam) if pred_class == 1 else 0.0   # severity only meaningful for predicted-defective images

    fname = safe_name(path, i)
    shutil.copy(path, TEST_IMG_DIR / fname)   # copy the real image into the export (the original Colab path won't exist locally)

    img_arr = unnormalize(x)
    fig, ax = plt.subplots(figsize=(3, 3))
    ax.imshow(img_arr); ax.imshow(cam, cmap="jet", alpha=0.45); ax.axis("off")
    plt.tight_layout(pad=0)
    plt.savefig(GRADCAM_DIR / fname, dpi=100, bbox_inches="tight", pad_inches=0)
    plt.close(fig)

    results.append({
        "image_file": fname, "gradcam_file": fname,
        "true": int(y), "pred": int(pred_class),
        "p_defective_raw": float(probs[1]), "severity": sev, "mc_std": float(mc_std[i]),
        "aleatoric": float(aleatoric_uncertainty[i]), "epistemic": float(epistemic_uncertainty[i]),
        "ood_distance": float(test_maha[i]),
    })
print(f"Exported {len(results)} test images + Grad-CAM overlays to {TEST_IMG_DIR} and {GRADCAM_DIR}")

# recompute calibrated confidence properly using the fitted temperature T (section 10)
test_logits_list = []
with torch.no_grad():
    for x, y, p in test_dl:
        test_logits_list.append(model(x.to(DEVICE)).cpu())
test_logits_cat = torch.cat(test_logits_list)
cal_probs = torch.softmax(test_logits_cat / T, dim=1)[:, 1].numpy()
for i, r in enumerate(results):
    r["p_defective_calibrated"] = float(cal_probs[i])
    uncertainty = 1 - abs(2 * r["p_defective_calibrated"] - 1)   # 0 = very confident either way, 1 = maximally unsure (p=0.5)
    r["priority_score"] = r["severity"] * (0.5 * uncertainty + 0.5 * (r["mc_std"] / (mc_std.max() + 1e-8)))

import pandas as pd
df = pd.DataFrame(results).sort_values("priority_score", ascending=False)
(WORK / "outputs").mkdir(exist_ok=True)
df.to_csv(WORK / "outputs" / "priority_ranked_results.csv", index=False)
print("Top 10 highest-priority images for manual review:")
display(df.head(10)[["image_file", "pred", "severity", "p_defective_calibrated", "mc_std", "priority_score"]])


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4.5))
axs[0].hist(df["priority_score"], bins=30, edgecolor="black")
axs[0].set_xlabel("priority score"); axs[0].set_ylabel("count"); axs[0].set_title("Inspection-priority score distribution")

defective_df = df[df["pred"] == 1]
axs[1].scatter(defective_df["severity"], defective_df["mc_std"], alpha=0.5)
axs[1].set_xlabel("Grad-CAM severity proxy"); axs[1].set_ylabel("MC-Dropout uncertainty (std)")
axs[1].set_title("Severity vs. uncertainty (predicted-defective images)")
plt.tight_layout()
plt.savefig(WORK / "outputs" / "visualizations" / "severity_priority_distributions.png", dpi=120)
plt.show()


## 13. Selective prediction — reject low-confidence cases instead of forcing an answer

Images below a calibrated-confidence threshold get flagged "needs manual review" instead of a forced yes/no. Shows the accuracy/coverage trade-off: accuracy on the images the model DOES commit to should be noticeably higher than overall accuracy.

In [ ]:
REJECT_THRESHOLD = 0.7   # calibrated confidence in the predicted class must exceed this, else reject

pred_conf = np.maximum(cal_probs, 1 - cal_probs)   # calibrated confidence in whichever class was predicted
accept_mask = pred_conf >= REJECT_THRESHOLD

coverage = accept_mask.mean()
acc_accepted = accuracy_score(all_y[accept_mask], all_preds[accept_mask]) if accept_mask.sum() else float("nan")
acc_rejected = accuracy_score(all_y[~accept_mask], all_preds[~accept_mask]) if (~accept_mask).sum() else float("nan")

print(f"Reject threshold: {REJECT_THRESHOLD}")
print(f"Coverage (fraction auto-decided): {coverage:.3f}")
print(f"Accuracy on accepted (auto-decided) predictions: {acc_accepted:.3f}")
print(f"Accuracy on rejected (flagged for manual review): {acc_rejected:.3f}")
print(f"Overall accuracy without selective prediction: {test_acc:.3f}")


# sweep across thresholds for a full coverage-vs-accuracy picture (not just one fixed threshold)
sweep_thresholds = np.arange(0.5, 1.0, 0.05)
sweep_coverage, sweep_acc = [], []
for t in sweep_thresholds:
    m = pred_conf >= t
    sweep_coverage.append(float(m.mean()))
    sweep_acc.append(float(accuracy_score(all_y[m], all_preds[m])) if m.sum() else None)


### Coverage vs. accuracy trade-off (selective prediction, swept across thresholds)

In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 4.5))
ax1.plot(sweep_thresholds, sweep_coverage, "o-", color="tab:blue", label="coverage")
ax1.set_xlabel("confidence threshold"); ax1.set_ylabel("coverage (fraction auto-decided)", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(sweep_thresholds, sweep_acc, "o-", color="tab:red", label="accuracy on accepted")
ax2.set_ylabel("accuracy on accepted predictions", color="tab:red")
plt.title("Selective prediction: raising the threshold trades coverage for accuracy")
plt.tight_layout()
plt.savefig(WORK / "outputs" / "visualizations" / "selective_prediction_sweep.png", dpi=120)
plt.show()


## 14. Grad-CAM sanity check

Proves the explanations are genuinely tied to what the model learned, not a generic edge-detector artifact. We randomize the LAST layer's weights (a standard version of the Adebayo et al. 2018 cascading randomization test) and show the heatmap changes/degrades - if Grad-CAM still looked identical on a randomized model, that would mean it isn't really explaining anything.

In [ ]:
import copy

sanity_model = copy.deepcopy(model)
with torch.no_grad():
    for p in sanity_model.fc.parameters():
        nn.init.normal_(p, mean=0.0, std=0.1)   # randomize the final classifier layer

sanity_cam_fn = GradCAM(sanity_model, sanity_model.layer4[-1])

x, y, path = test_ds[0]
xb = x.unsqueeze(0).to(DEVICE)
cam_real, pred_real, probs_real = gradcam(xb)
cam_random, pred_random, probs_random = sanity_cam_fn(xb)

fig, axs = plt.subplots(1, 3, figsize=(12, 4))
img = unnormalize(x)
axs[0].imshow(img); axs[0].set_title("original"); axs[0].axis("off")
axs[1].imshow(img); axs[1].imshow(cam_real, cmap="jet", alpha=0.45); axs[1].set_title("Grad-CAM (trained model)"); axs[1].axis("off")
axs[2].imshow(img); axs[2].imshow(cam_random, cmap="jet", alpha=0.45); axs[2].set_title("Grad-CAM (randomized final layer)"); axs[2].axis("off")
plt.tight_layout()
plt.savefig(WORK / "outputs" / "visualizations" / "gradcam_sanity_check.png", dpi=120)
plt.show()
print("If the randomized-model heatmap looks noticeably different/noisier than the trained one, the sanity check passes -")
print("it shows Grad-CAM is actually reading learned weights, not just highlighting generic image edges.")


## 15. Export everything

Saves the trained model, all metrics, the calibration temperature, and visualizations to Drive, then offers a browser download of a zipped bundle.

In [ ]:
metrics = {
    "test_accuracy": float(test_acc),
    "roc_auc": float(roc_auc),
    "confusion_matrix": cm.tolist(),
    "roc_curve": {"fpr": fpr.tolist(), "tpr": tpr.tolist()},
    "training_history": history,
    "ece_before_calibration": float(ece_before),
    "ece_after_calibration": float(ece_after),
    "fitted_temperature": float(T),
    "reliability_diagram": {
        "bin_edges": bin_edges,
        "raw": {"confidence": raw_bin_conf, "accuracy": raw_bin_acc, "count": raw_bin_count},
        "calibrated": {"confidence": cal_bin_conf, "accuracy": cal_bin_acc, "count": cal_bin_count},
    },
    "mc_dropout_std_summary": {"min": float(mc_std.min()), "median": float(np.median(mc_std)), "max": float(mc_std.max())},
    "mc_dropout_std_values": mc_std.tolist(),
    "selective_prediction": {
        "reject_threshold": REJECT_THRESHOLD,
        "coverage": float(coverage),
        "accuracy_accepted": float(acc_accepted),
        "accuracy_rejected": float(acc_rejected),
        "threshold_sweep": {"thresholds": sweep_thresholds.tolist(), "coverage": sweep_coverage, "accuracy": sweep_acc},
    },
    "n_train": len(train_p), "n_val": len(val_p), "n_test": len(test_p),
    "dataset_stats": {
        "train": {"total": len(train_p), "defective": int(sum(train_y)), "non_defective": int(len(train_y) - sum(train_y))},
        "val": {"total": len(val_p), "defective": int(sum(val_y)), "non_defective": int(len(val_y) - sum(val_y))},
        "test": {"total": len(test_p), "defective": int(sum(test_y)), "non_defective": int(len(test_y) - sum(test_y))},
    },
    "uncertainty_decomposition": {
        "aleatoric": {"min": float(aleatoric_uncertainty.min()), "median": float(np.median(aleatoric_uncertainty)), "max": float(aleatoric_uncertainty.max())},
        "epistemic": {"min": float(epistemic_uncertainty.min()), "median": float(np.median(epistemic_uncertainty)), "max": float(epistemic_uncertainty.max())},
    },
    "ood_mahalanobis": {
        "train_reference_median": float(np.median(train_maha)),
        "test_median": float(np.median(test_maha)),
        "test_max": float(test_maha.max()),
        "top5_furthest_test_images": [str(all_paths[i]) for i in top_ood_idx],
    },
}
with open(WORK / "outputs" / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))

EXPORT_ZIP = "/content/raildefect_export.zip"
shutil.make_archive("/content/raildefect_export", "zip", WORK / "outputs")
from google.colab import files
files.download(EXPORT_ZIP)
print("\nDownloaded raildefect_export.zip (also saved to Drive at", WORK / "outputs", ")")
print("Contains: best_model.pt, metrics.json, priority_ranked_results.csv, visualizations/")
